In [ ]:
!pip install datasets
!pip install mwclient
!pip install bitsandbytes
!pip install -U bitsandbytes
!pip install huggingface
!pip install beautifulsoup4
!pip install peft
!pip install transformers

In [ ]:
model ="mistralai/Mistral-Nemo-Instruct-2407"
from huggingface_hub import notebook_login
notebook_login() #obtain access to the gated model through your account registered on the huggingface website, and then enter your account's access token

In [ ]:
import requests
from bs4 import BeautifulSoup
import os
import re
from peft import prepare_model_for_kbit_training, get_peft_model, LoraConfig
import mwclient
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments
import torch
def scrape_qri_wiki():
    pages = []
    user_agent = 'QuaLLM (qua@llm.org)'
    site = mwclient.Site('wiki.qri.org', path='/w139/', clients_useragent=user_agent)
    for page in site.allpages():
        x = page.text()
        if "#REDIRECT" not in x:
          x = re.sub(r"''(.*?)''", r"\1", x)
           # Remove headers
          x = re.sub(r"==+.*?==+", "", x)

          # Remove internal links and keep the text
          x = re.sub(r"\[\[(?:[^|\]]*\|)?([^\]]+)\]\]", r"\1", x)

            # Remove external links
          x = re.sub(r"\[https?://[^ ]+ ([^\]]+)\]", r"\1", x)

          x = re.sub(r'\{\{.*?\}\}', '', x)
            # Remove reference tags
          x = re.sub(r"<ref>.*?</ref>", "", x)

            # Remove any remaining HTML tags
          x = re.sub(r"<.*?>", "", x)
          x = re.sub(r"\[\[File:[^\]]*\]\]", "", x)

            # Strip excess whitespace
          x = re.sub(r"\s+", " ", x).strip()
          pages.append(x)
    return pages
def prepare_dataset(pages):
    tokenizer = AutoTokenizer.from_pretrained(model)
    tokenizer.pad_token = tokenizer.eos_token
    tokenized_data = tokenizer(articles, padding=True, truncation=True, return_tensors="pt")
    tokenized_data["labels"] = tokenized_data["input_ids"].clone()
    tokenized_data["attention_mask"] = tokenized_data["attention_mask"]
    dataset = Dataset.from_dict(tokenized_data)  # Convert to Dataset object
    return dataset, tokenizer
def fine_tune_llm(tokenized_data, tokenizer):
    bnb_config = BitsAndBytesConfig(
                                load_in_4bit=True,
                                bnb_4bit_use_double_quant=True,
                                bnb_4bit_quant_type="nf4",
                                bnb_4bit_compute_dtype=torch.bfloat16,
                               )
    model = AutoModelForCausalLM.from_pretrained(model,quantization_config=bnb_config,device_map="auto")
    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)
    config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
        "lm_head",
    ],
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)
    model = get_peft_model(model, config)

    training_args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=9,
        per_device_train_batch_size=3,
        save_steps=10_000,
        save_total_limit=2,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_data,
        tokenizer=tokenizer
    )

    trainer.train()
    model.save_pretrained("./fine_tuned_qri_model")
    tokenizer.save_pretrained("./fine_tuned_qri_model")

print("Scraping QRI Wiki...")
articles = scrape_qri_wiki()
print(f"Scraped {len(articles)} articles.")

print("Preparing dataset...")
tokenized_data, tokenizer = prepare_dataset(articles)

print("Fine-tuning the language model...")
fine_tune_llm(tokenized_data, tokenizer)
print("Model fine-tuned and saved!")

In [ ]:
def send_prompt(prompt, model_path="./fine_tuned_qri_model"):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path)
    inputs = tokenizer(prompt, return_tensors="pt")
    print("generating")
    outputs = model.generate(inputs.input_ids, max_length=500,attention_mask=inputs["attention_mask"],temperature=0.3,do_sample=True)
    print("generated")
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
prompt = "What is Qualia Research Institute?"
response = send_prompt(prompt)
print(f"Prompt Response: {response}")